# Spark Architecture

**Module 2 | Fundamentals Training**

Before you start writing pipelines, you need to understand **how Spark really executes your code**. This module explains why some operations are slow, where OOM errors come from, and why your code "doesn't work" despite no errors — because it hasn't executed yet.

## Topics

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Driver & Executors | Cluster architecture, work distribution |
| 2 | Lazy Evaluation | Transformations vs Actions, when code executes |
| 3 | DAG, Stages, Shuffles | How Spark plans and divides work |
| 4 | Catalyst Optimizer + AQE | How Spark optimizes queries |
| 5 | Practical demo | Observing what triggers execution + Spark UI |


## Learning Objectives

After this module you will be able to:

- **Explain** the role of the Driver and Executors in a Spark cluster
- **Distinguish** between a transformation (lazy) and an action (eager) and give examples
- **Predict** when a notebook cell will actually start executing
- **Understand** what a shuffle is and why it is expensive
- **Describe** how the Catalyst Optimizer and AQE improve performance
- **Find** basic execution information in the Spark UI


## Setup

In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

print(f"Catalog: {CATALOG}")
print(f"Spark:   {spark.version}")

---
## 1. Driver & Executors — cluster architecture

When you run a cell in a notebook, Databricks does not execute it on your laptop. The code goes to a **Spark cluster**, which consists of:

<img src="../../assets/images/spark_cluster_architecture.svg" width="860">

### Driver
- One process per cluster — coordinates all work
- Holds `SparkSession`, builds the query plan, schedules tasks
- If the Driver crashes → the job crashes
- **Your Python code runs on the Driver** — loops, list comprehensions, `print()` — all on the Driver

### Executors
- Many processes on worker nodes — perform the actual computation
- Each Executor processes one or more **partitions** in parallel
- Results return to the Driver (e.g., with `collect()`) or are written to storage

### Partition — unit of parallelism
Data in Spark is divided into **partitions** (~128 MB each). One partition = one task on one core of one Executor.

<img src="../../assets/images/spark_partitions.svg" width="860">

> **Key principle:** Spark does not move data to the computation — it moves small code to where the data is (data locality). That is why it scales to petabytes.


---
## 2. Lazy Evaluation — code that doesn't run immediately

This is **the most important concept in Spark** and the source of the most confusion for beginners.

### Transformations vs Actions

| Type | Examples | What happens |
|------|----------|-------------|
| **Transformation** (lazy) | `filter()`, `select()`, `groupBy()`, `join()`, `withColumn()`, `orderBy()` | **Nothing!** Spark records the plan but does not compute |
| **Action** (eager) | `count()`, `show()`, `collect()`, `write()`, `display()`, `first()` | Spark runs the ENTIRE plan and delivers the result |

### Why is laziness good?
Spark sees the entire plan before execution and can:
- move a filter **before** an expensive join (predicate pushdown)
- remove columns you don't need (column pruning)
- merge multiple `filter()` calls into a single pass over the data
- choose the optimal join strategy (broadcast vs shuffle)


In [0]:
# ============================================================
# DEMO: Lazy Evaluation — when does code ACTUALLY execute?
# ============================================================

print("Step 1: Loading data — does anything get computed?")
t0 = time.time()

# This line does NOT read the file yet — it only registers the plan
df = spark.table("samples.nyctaxi.trips")

t1 = time.time()
print(f"  spark.table() took: {t1-t0:.3f}s — almost nothing! (lazy)")
print(f"  Object type: {type(df)}\n")


In [0]:
print("Step 2: Adding transformations — does anything get computed?")
t0 = time.time()

# Each transformation ONLY extends the logical plan — zero computation
df2 = df \
    .filter(F.col("fare_amount") > 10) \
    .filter(F.col("trip_distance") > 1.0) \
    .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2)) \
    .select("pickup_zip", "dropoff_zip", "fare_amount", "trip_distance", "fare_per_mile")

t1 = time.time()
print(f"  4 transformations took: {t1-t0:.3f}s — still almost nothing! (lazy)\n")


In [0]:
print("Step 3: First ACTION — now Spark actually computes!")
t0 = time.time()

# count() is an action — triggers execution of the ENTIRE plan
row_count = df2.count()

t1 = time.time()
print(f"  count() took: {t1-t0:.2f}s — now ALL DATA was processed")
print(f"  Result: {row_count:,} rows")


In [0]:
print("Step 4: display() = another action = ANOTHER full plan execution!")
t0 = time.time()

# display() is also an action — Spark recomputes from scratch (without cache)
display(df2.limit(10))

t1 = time.time()
print(f"  display(limit 10) took: {t1-t0:.2f}s")
print()
print("Conclusion: if you want to use df2 multiple times → do df2.cache() or df2.persist()!")


In [0]:
# Preview of the execution plan — what Spark PLANS to do (before executing)
print("=== Logical plan (without optimization) ===")
df2.explain(mode="simple")


In [0]:
# Full plan with Catalyst optimizations
print("=== Physical plan (after Catalyst optimization) ===")
df2.explain(mode="formatted")


In [0]:
# To check Spark UI: In Databricks, click "View" > "Spark UI" in the notebook menu bar.
display(df2)

---
## 3. DAG, Stages and Shuffles

When an action triggers execution, Spark builds a **DAG** (Directed Acyclic Graph) — a graph of all the steps, splits it into **stages** and sends them to Executors.

### When does a new Stage appear?

Stage boundary = **shuffle** = the moment when data must be redistributed between Executors.

| Operation | Shuffle? | Explanation |
|-----------|----------|-------------|
| `filter()`, `select()`, `withColumn()` | No (narrow) | Each partition is independent |
| `groupBy().agg()` | Yes (wide) | Data by key must reach the same Executor |
| `join()` (large tables) | Yes (wide) | Matching keys must be co-located |
| `orderBy()` / `sort()` | Yes (wide) | Global sort requires collecting data |
| `repartition(n)` | Yes | Explicit redistribution |
| `coalesce(n)` | No | Reducing partitions without full shuffle |
| Broadcast join | No | Small table is sent to every Executor |

> **Shuffle is expensive** because it requires writing data to disk and transmitting over the network. Minimizing shuffles = the main Spark optimization technique.


<img src="../../assets/images/spark_dag_stages_shuffle.svg" width="900">

In [0]:
# Demo: narrow vs wide transformation
# Observe in Spark UI how many Stages each query has

df_trips = spark.table("samples.nyctaxi.trips")

# NARROW — no shuffle — 1 Stage
print("Narrow (filter + select) — should be 1 Stage:")
result_narrow = df_trips \
    .filter(F.col("fare_amount") > 20) \
    .select("pickup_zip", "dropoff_zip", "fare_amount")


In [0]:
display(result_narrow)  # action triggers

# Done! Check Spark UI → Jobs → last job → how many Stages?

In [0]:
# WIDE — shuffle on groupBy — 2+ Stages
print("Wide (groupBy + agg) — should be 2+ Stages:")

result_wide = df_trips \
    .groupBy("pickup_zip") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    ) \
    .orderBy(F.desc("trip_count"))


In [0]:
display(result_wide)
# Done! Compare in Spark UI how many Stages this time.

---
## 4. Catalyst Optimizer + Adaptive Query Execution (AQE)

### Catalyst Optimizer (compile-time)
Before Spark sends anything to Executors, Catalyst **rewrites the logical plan** into an optimal physical plan:

| Optimization | Example |
|--------------|---------|
| **Predicate Pushdown** | `df.join(other).filter(col > 10)` → filter is placed BEFORE the join |
| **Column Pruning** | `df.select("a","b").filter(...)` → Spark doesn't read other columns from Parquet |
| **Constant Folding** | `filter(col > 100 - 50)` → Spark computes `50` at compile-time |
| **Join Reordering** | Smaller table as build side in hash join |

### Adaptive Query Execution — AQE (runtime)
AQE works **during** execution — it improves the plan based on actual statistics:

| AQE Feature | What it does |
|-------------|-------------|
| **Dynamic shuffle partitions** | Reduces 200 → X partitions based on actual data size |
| **Broadcast join conversion** | If after a filter a table turns out to be small → converts to broadcast |
| **Skew handling** | Detects and splits uneven (skewed) partitions |

> Thanks to Catalyst and AQE **the same SQL and PySpark query generates an identical plan**. There is no performance difference between `spark.sql("SELECT...")` and the DataFrame API.


<img src="../../assets/images/spark_catalyst_pipeline.svg" width="900">

In [0]:
# Check if AQE is enabled (True by default since Spark 3.0)
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")
aqe_coalesce = spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")
broadcast_thresh = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

print(f"AQE enabled:                {aqe_enabled}")
print(f"AQE coalesce partitions:    {aqe_coalesce}")
print(f"Auto broadcast threshold:   {broadcast_thresh} bytes")
print(f"Shuffle partitions (static):{spark.conf.get('spark.sql.shuffle.partitions')}")

In [0]:
# Comparison: PySpark API vs SQL — is the execution plan identical?
df_trips = spark.table("samples.nyctaxi.trips")

# PySpark DataFrame API
plan_pyspark = df_trips \
    .filter(F.col("fare_amount") > 15) \
    .groupBy("pickup_zip") \
    .agg(F.avg("fare_amount").alias("avg_fare")) \
    .explain(mode="simple")

print("\n" + "="*50)

# SQL — identical plan after Catalyst compilation
spark.sql("""
    SELECT pickup_zip, AVG(fare_amount) AS avg_fare
    FROM samples.nyctaxi.trips
    WHERE fare_amount > 15
    GROUP BY pickup_zip
""").explain(mode="simple")

---
## 5. Spark UI — where to find execution information

Spark UI is a built-in diagnostic interface available **directly from the cluster in Databricks**.

### How to open Spark UI:
1. Click **Compute** in the left menu
2. Select your cluster
3. Click **Spark UI** (tab at the top)

### Key Spark UI tabs:

| Tab | What it shows | When to use |
|-----|---------------|-------------|
| **Jobs** | List of jobs — time, status, number of Stages | First stop when debugging |
| **Stages** | Details of each Stage: tasks, shuffle read/write, time | When a Stage is slow |
| **SQL / DataFrame** | Query plan with timing for each node | Optimizing SQL queries |
| **Storage** | Cached DataFrames (size, fraction cached) | Cache management |
| **Executors** | Memory and CPU usage per Executor | OOM debugging |

### What to look for in Spark UI when a job is slow:
1. **Long Stage** → check shuffle read/write — large shuffle = bottleneck
2. **Uneven task times** → data skew (some partitions much larger)
3. **High GC time** → Executor out of memory → increase executor memory or reduce data
4. **Few tasks** → few partitions → data is not being processed in parallel


In [0]:
# Run this query, then open Spark UI → Jobs → last job → click on a Stage
# You will see: how many tasks, how long, shuffle read/write

result = spark.table("samples.nyctaxi.trips") \
    .groupBy("pickup_zip", "dropoff_zip") \
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    ) \
    .filter(F.col("trips") >= 10) \
    .orderBy(F.desc("total_revenue"))

display(result.limit(15))

print("\nNow open Spark UI → Jobs → check how many Stages, how many tasks, was there a shuffle.")


---
## Bonus — Cache and Persist

If you use the same DataFrame multiple times (e.g., a filter + two different joins), without cache Spark recomputes it every single time.


In [0]:
# Example: without cache vs with cache
df_base = spark.table("samples.nyctaxi.trips") \
    .filter(F.col("fare_amount").between(5, 100)) \
    .filter(F.col("trip_distance") > 0.5)

# WITHOUT CACHE — every action = full file scan
t0 = time.time()
c1 = df_base.count()
t1 = time.time()
print(f"Without cache — count: {c1:,} rows | time: {t1-t0:.2f}s")

# WITH CACHE — first call builds the cache, subsequent ones read from memory
df_base.cache()  # registers the cache intent (lazy!)

t0 = time.time()
c2 = df_base.count()  # this triggers building the cache
t1 = time.time()
print(f"With cache — 1st count: {c2:,} rows | time: {t1-t0:.2f}s  (building cache)")

t0 = time.time()
c3 = df_base.count()  # now from cache
t1 = time.time()
print(f"With cache — 2nd count: {c3:,} rows | time: {t1-t0:.2f}s  (from memory — faster!)")


In [0]:
# Always release the cache after use — Executor memory is limited!
df_base.unpersist()
print("Cache released.")


---
## Summary

### Key principles to remember:

| Principle | What it means in practice |
|-----------|--------------------------|
| **Lazy Evaluation** | `filter()`, `select()`, `withColumn()` — compute nothing. `count()`, `display()`, `write()` — trigger everything |
| **Shuffle is expensive** | `groupBy`, `join`, `orderBy` = shuffle. Minimize them. Use broadcast for small tables |
| **PySpark = SQL** | Catalyst compiles both to the same plan — no performance difference |
| **AQE handles this automatically** | Databricks optimizes shuffle partitions and joins at runtime — you don't need to do anything |
| **Cache when reusing** | One `df.cache()` + one cold run = fast subsequent actions |
| **Spark UI** | Every slow job → Spark UI → Jobs → Stage → tasks |

```

> **Next module:** M03 — Medallion Architecture + Ingestion (loading data)
